# 02 · Camada Silver
**Entrada:** tabelas `bronze.tb_*` (dado bruto, todas as colunas como `string`) · **Saída:** tabelas `silver.tb_*`

## Resumo
Este notebook transforma a camada Bronze em Silver: dados **limpos, tipados, com colunas em português e sem duplicatas**. As tabelas Bronze **não são alteradas** (exigência do enunciado), apenas lidas.

> **Escopo:** este notebook cobre **5 das 7** tabelas Silver do enunciado. `silver.tb_financeiro_filmes` e `silver.tb_cotacao_dolar` dependem da cotação do dólar e estão registradas em **Pendências** (final do notebook).

## Estrutura de cada seção
**Leitura → Exploração → Transformação → Validação.**
1. **Exploração:** mede a sujeira real da origem *antes* de escrever a regra, para que cada regra tenha um motivo observável.
2. **Transformação:** aplica as regras e grava a tabela Silver.
3. **Validação:** confere a tabela gravada, no espírito das *expectations* das aulas (`FAIL` = `assert`, interrompe o pipeline; `QUIET` = apenas reporta).

## Decisões de projeto que valem para o notebook inteiro
- **Conversão segura de tipos.** O serverless roda com modo ANSI: um `cast` direto de `"Unknown"` para número **interrompe o pipeline**. Por isso usamos `try_cast` ou validamos o formato com regex antes de converter, e o texto inválido vira `NULL`.
- **Chave `id` numérica.** Com *column shift*, texto cai até na coluna de chave. Linhas cujo `id` não é numérico não representam filme algum e são descartadas.
- **Funções compartilhadas (seção 0).** A lógica de deduplicação e de split + explode aparece em mais de uma tabela. Escrevê-la uma vez evita divergência entre cópias.

## 0. Setup

In [0]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "atividade_2_cinedata"
spark.sql(f"USE CATALOG {CATALOGO}")

In [0]:
ID_NUMERICO = r"^[0-9]+$"   # chave válida de filme

# Domínio fechado de gêneros (TMDB).
GENEROS = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
           "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
           "Science Fiction", "TV Movie", "Thriller", "War", "Western"]
GENEROS_LOWER = [g.lower() for g in GENEROS]

# Textos que representam ausência de dado nas colunas de pessoas e empresas
PLACEHOLDERS = ["", "null", "none", "nan", "n/a", "na", "nenhum", "[]", "unknown",
                "desconhecido", "não informado", "nao informado", "-"]


def deduplicar_mais_recente(df, chave="id"):
    preenchidas = sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in df.columns)
    janela = Window.partitionBy(chave).orderBy(F.col("ingestion_datetime").desc(), preenchidas.desc())
    return (df.where(F.col(chave).isNotNull())
              .withColumn("rn", F.row_number().over(janela))
              .where("rn = 1").drop("rn"))


def separar_e_explodir(df, coluna, alias="valor"):
    # Split + explode: um registro por filme e por valor. Os separadores vírgula, ponto e vírgula e barra vertical coexistem na origem. Valores nulos desaparecem no explode.
    return (df.select(F.col("id").alias("id_filme"),
                      F.explode(F.split(F.col(coluna), r"[,;|]")).alias("bruto"))
              .select("id_filme",
                      F.trim(F.regexp_replace("bruto", r'''^[\s"'\[\]]+|[\s"'\[\]]+$''', "")).alias(alias)))

## 1. `silver.tb_info_filmes`
**Origem:** `bronze.tb_movies_info` · **Grão:** um registro por filme.

### O que a exploração da origem mostrou
- **Duplicatas:** 213.192 linhas para 97.611 filmes (cerca de 2,2 registros por filme: cargas repetidas na Bronze e/ou duplicatas no CSV).
- **Status:** variação de caixa (`Released`, `RELEASED`, `released`), hífens (`In-Production`) e, em poucas linhas, uma **data no lugar do status** (*column shift*).
- **Data de lançamento:** três formatos válidos (`2016-02-09`, `04-25-2018`, `10/05/2018`) e linhas em que um **título** caiu na coluna de data.
- **Duração:** valores como `en` (código de idioma) na coluna numérica.
- **`tconst`** (id do IMDb) existe na origem, mas **não consta no mapeamento** do enunciado.

In [0]:
df_info_bronze = spark.table("bronze.tb_movies_info")

### 1.1 Exploração
Medimos os problemas do dataset: duplicatas, valores de `status`, padrões de data e durações que não são inteiros.

In [0]:
print("Linhas:", df_info_bronze.count(), "| ids distintos:", df_info_bronze.select("id").distinct().count())

# Valores de status como vieram (revela variações de caixa, hífens e lixo)
display(df_info_bronze.groupBy("status").count().orderBy(F.desc("count")))

# Formatos de data presentes na origem, com um exemplo de cada
padrao_data = F.regexp_replace(F.regexp_replace("release_date", "[0-9]", "9"), "[A-Za-z]", "A")
display(df_info_bronze.groupBy(padrao_data.alias("padrao_data"))
        .agg(F.count("*").alias("qtd"), F.first("release_date").alias("exemplo"))
        .orderBy(F.desc("qtd")))

# Duração que não é número (column shift)
display(df_info_bronze.where(F.expr("try_cast(runtime AS INT) IS NULL AND runtime IS NOT NULL"))
        .select("runtime").distinct().limit(20))

#### Diagnóstico das datas ambíguas
`10/05/2018` pode ser 10 de maio ou 5 de outubro. Nas datas **não ambíguas** (um dos números passa de 12) o formato se resolve sozinho. Esta célula conta, para cada padrão com separador, em qual posição aparece um número maior que 12:
- `primeira_parte_maior_12 > 0` e `segunda_parte_maior_12 = 0` → o formato é **dia primeiro** (`dd/MM`);
- o contrário → **mês primeiro** (`MM/dd`).

O resultado deve ser conferido contra a ordem da lista `formatos` na transformação (1.2): a ordem só decide as datas ambíguas.

In [0]:
def diagnostico_ambiguidade(regex):
    base = df_info_bronze.where(F.col("release_date").rlike(regex))
    p1 = F.expr("try_cast(substring(release_date, 1, 2) AS INT)")
    p2 = F.expr("try_cast(substring(release_date, 4, 2) AS INT)")
    return base.agg(
        F.count("*").alias("total"),
        F.sum((p1 > 12).cast("int")).alias("primeira_parte_maior_12"),
        F.sum((p2 > 12).cast("int")).alias("segunda_parte_maior_12"),
        F.sum(((p1 <= 12) & (p2 <= 12)).cast("int")).alias("ambiguas"))

print("Padrão com barra (99/99/9999):");  display(diagnostico_ambiguidade(r"^\d{2}/\d{2}/\d{4}$"))
print("Padrão com hífen (99-99-9999):");  display(diagnostico_ambiguidade(r"^\d{2}-\d{2}-\d{4}$"))

### 1.2 Transformação
**Decisões:**
- Deduplicação por ingestão mais recente;
- Status é corrigido em duas etapas: 1. normaliza e 2. traduz
- Data é corrigida para um formato só, caso a correção seja impossível, se torna `NULL`
- Coluna `ano_lançamento` a partir de `data_lançamento`

In [0]:
# 1) Deduplicação
df_dedup = deduplicar_mais_recente(df_info_bronze, "id")

# 2) Status: normaliza e traduz. Rótulos idênticos aos do enunciado.
status_norm = F.trim(F.regexp_replace(
    F.regexp_replace(F.lower(F.col("status")), "[^a-z]+", " "), "\\s+", " "))

traducao = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado",
    "cancelled": "Cancelado",    
}
STATUS_NAO_INFORMADO = "Não Informado"
mapa = F.create_map(*[F.lit(x) for kv in traducao.items() for x in kv])
status_pt = F.coalesce(mapa[status_norm], F.lit(STATUS_NAO_INFORMADO))

# 3) Data de lançamento: tenta cada padrão em ordem; NULL só se nenhum funcionar.
formatos = ["yyyy-MM-dd", "MM-dd-yyyy", "dd/MM/yyyy", "MM/dd/yyyy", "yyyy/MM/dd"]
data_lancamento = F.coalesce(*[
    F.expr(f"CAST(try_to_timestamp(trim(release_date), '{f}') AS DATE)") for f in formatos])

df_info_silver = (df_dedup.select(
    F.col("id").cast("string").alias("id_filme"),
    F.col("title").alias("titulo"),
    F.col("original_title").alias("titulo_original"),
    data_lancamento.alias("data_lancamento"),
    F.expr("try_cast(runtime AS INT)").alias("duracao_minutos"),
    F.col("original_language").alias("idioma_original"),
    status_pt.alias("status_filme"),
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao"),
).withColumn("ano_lancamento", F.year("data_lancamento")))
# A coluna tconst (IMDb) não consta no mapeamento do enunciado e foi descartada.

df_info_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_info_filmes")

### 1.3 Validação

In [0]:
s = spark.table("silver.tb_info_filmes")

# FAIL: unicidade e completude (a Silver deve ter exatamente os filmes distintos da Bronze)
filmes_bronze = df_info_bronze.where("id IS NOT NULL").select("id").distinct().count()
assert s.count() == s.select("id_filme").distinct().count(), "Há filmes duplicados"
assert s.count() == filmes_bronze, f"Silver tem {s.count()} filmes, Bronze tem {filmes_bronze}"

# FAIL: status só pode ter os valores do domínio do enunciado
STATUS_VALIDOS = set(traducao.values()) | {STATUS_NAO_INFORMADO}
assert s.where(~F.col("status_filme").isin(list(STATUS_VALIDOS))).count() == 0, "Status fora do domínio"

# QUIET: apenas reporta
print("Filmes:", s.count())
print("Datas nulas:", s.where("data_lancamento IS NULL").count(),
      "| Durações nulas:", s.where("duracao_minutos IS NULL").count())
display(s.groupBy("status_filme").count().orderBy(F.desc("count")))
display(s.where("data_lancamento IS NULL").select("id_filme", "titulo").limit(10))

## 2 · `silver.tb_metricas_engajamento`
**Origem:** `bronze.tb_movies_metrics` · **Grão:** um registro por filme.

### O que a exploração da origem mostrou
- **Duplicatas:** 206.144 linhas para 95.115 filmes. Todos os `id` são numéricos.
- **Popularidade:** usa ponto **ou** vírgula como separador decimal (`72.735` e `154,34`), nunca os dois juntos, e o TMDB usa 3 casas decimais. Portanto o ponto não é separador de milhar e basta trocar a vírgula por ponto.
- **Column shift:** textos de sinopse, nomes de pessoas, idiomas e gêneros espalhados pelas colunas numéricas.
- **Erro de escala:** milhares de `vote_average` acima de 10 (máximo 100) e algumas `averageRating` acima de 10.
- **Negativos:** nenhum na base atual, mas a regra é exigência do enunciado e permanece.

In [0]:
df_metricas_bronze = spark.table("bronze.tb_movies_metrics")

### 2.1 Exploração

In [0]:
print("ids não numéricos:", df_metricas_bronze.where(~F.col("id").rlike(ID_NUMERICO) | F.col("id").isNull()).count())
print("ids numéricos distintos:", df_metricas_bronze.where(F.col("id").rlike(ID_NUMERICO)).select("id").distinct().count())

# Resumo por coluna 
linhas = []
for coluna in ["popularity", "vote_average", "vote_count", "averageRating", "numVotes"]:
    v = F.expr(f"try_cast(replace({coluna}, ',', '.') AS DOUBLE)")
    r = df_metricas_bronze.agg(
        F.count(F.when(F.col(coluna).isNotNull(), 1)).alias("preenchidos"),
        F.count(F.when(F.col(coluna).isNotNull() & v.isNull(), 1)).alias("nao_numericos"),
        F.count(F.when(v < 0, 1)).alias("negativos"),
        F.count(F.when(v > 10, 1)).alias("acima_de_10"),
        F.min(v).alias("min"), F.max(v).alias("max"),
    ).first()
    linhas.append((coluna, *r))
display(spark.createDataFrame(linhas,
    "coluna string, preenchidos long, nao_numericos long, negativos long, acima_de_10 long, min double, max double"))

# Formato dos valores de popularity (dígito vira 9): confirma que há um único separador por valor
padrao_pop = F.regexp_replace("popularity", "[0-9]", "9")
display(df_metricas_bronze.where(F.col("popularity").rlike(r"^[0-9.,\-]+$"))
        .groupBy(padrao_pop.alias("padrao_popularity")).count().orderBy(F.desc("count")).limit(15))

### 2.2 Transformação
**Decisões:**
- **Converter só o que tem cara de número.** A função `num` testa o valor **inteiro** com uma regex e só então converte. Remover "tudo que não é dígito" seria perigoso: com *column shift*, um fragmento como `866` com aspas sobrando (aparece na origem) ou um ano dentro de uma sinopse viraria número, **fabricando dado** a partir de lixo. Texto que não passa na regex vira `NULL`.
- **Vírgula decimal → ponto** (`154,34` → `154.34`), pelo motivo visto na exploração.
- **Notas fora de 0 a 10 → `NULL`** (inclui o erro de escala), sem tentar dividir por 10.
- **Negativos → `NULL`** em popularidade e contagens.
- **Contagens limitadas ao tamanho de `INT`**, para o cast não estourar no modo ANSI.
- **Nota 0 é mantida:** está dentro do intervalo do enunciado (0 a 10). Na prática, `nota = 0` com `votos = 0` significa "sem votos"; a regra de negócio não a invalida, então não a alteramos.

In [0]:
NUM = r"^-?[0-9]+([.,][0-9]+)?$"

def num(coluna):
    # Texto -> DOUBLE de forma segura: só converte se o valor INTEIRO (após trim) tem formato
    # numérico; qualquer outro texto (column shift) vira NULL. Vírgula decimal vira ponto.
    t = F.trim(F.col(coluna))
    return F.when(t.rlike(NUM), F.regexp_replace(t, ",", ".").cast("double"))

def nota(coluna):
    x = num(coluna)
    return F.when((x >= 0) & (x <= 10), x)

def positivo(coluna):
    x = num(coluna)
    return F.when(x >= 0, x)

def contagem(coluna):
    x = num(coluna)
    return F.when((x >= 0) & (x <= 2147483647), x.cast("int"))

# Chave: descarta linhas cujo id é lixo de column shift; depois, uma linha por filme
df_metricas = deduplicar_mais_recente(df_metricas_bronze.where(F.col("id").rlike(ID_NUMERICO)), "id")

df_metricas_silver = df_metricas.select(
    F.col("id").alias("id_filme"),
    positivo("popularity").alias("popularidade"),
    nota("vote_average").alias("nota_media_tmdb"),
    contagem("vote_count").alias("qtd_votos_tmdb"),
    nota("averageRating").alias("nota_media_imdb"),
    contagem("numVotes").alias("qtd_votos_imdb"),
)

df_metricas_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_metricas_engajamento")

### 2.3 Validação

In [0]:
s = spark.table("silver.tb_metricas_engajamento")

# FAIL: unicidade
assert s.count() == s.select("id_filme").distinct().count(), "Há filmes duplicados"

# FAIL: limites de negócio
lim = s.agg(F.max("nota_media_tmdb"), F.min("nota_media_tmdb"),
            F.max("nota_media_imdb"), F.min("nota_media_imdb"),
            F.min("popularidade"), F.min("qtd_votos_tmdb"), F.min("qtd_votos_imdb")).first()
assert lim[0] <= 10 and lim[1] >= 0 and lim[2] <= 10 and lim[3] >= 0, "Nota fora de 0-10"
assert lim[4] >= 0 and lim[5] >= 0 and lim[6] >= 0, "Valor negativo"

# QUIET: só reporta
print("Filmes:", s.count())
for coluna in s.columns[1:]:
    print(f"{coluna}: {s.where(F.col(coluna).isNull()).count()} nulos")
display(s.limit(10))

# 3 · `silver.tb_avaliacoes_usuarios`
**Origem:** `bronze.tb_movies_reviews` · **Grão:** uma avaliação por combinação filme + usuário + nota + comentário.

## O que a exploração da origem mostrou
- **Duplicatas integrais em volume:** 64.824 linhas, mas só 32.412 combinações distintas (exatamente a metade, o que é coerente com uma carga repetida na Bronze).
- **Comentário:** 15.552 `NULL`; nenhum só com espaços. Na amostra do CSV o texto literal `null` também aparecia como comentário.
- **Nota:** nenhum texto na coluna, mas a regra de 0 a 10 continua obrigatória.



In [0]:
df_reviews_bronze = spark.table("bronze.tb_movies_reviews")

### 3.1 Exploração

In [0]:
print("Linhas:", df_reviews_bronze.count(), "| ids distintos:", df_reviews_bronze.select("id").distinct().count())
print("ids não numéricos:", df_reviews_bronze.where(~F.col("id").rlike(ID_NUMERICO) | F.col("id").isNull()).count())

# Notas
v = F.expr("try_cast(replace(trim(nota), ',', '.') AS DOUBLE)")
display(df_reviews_bronze.agg(
    F.count(F.when(F.col("nota").isNotNull() & v.isNull(), 1)).alias("nao_numericas"),
    F.count(F.when(v < 0, 1)).alias("negativas"),
    F.count(F.when(v > 10, 1)).alias("acima_de_10"),
    F.min(v).alias("min"), F.max(v).alias("max")))

# Comentários mais comuns (revela 'null', vazios e espaços) e contagem de ausentes
display(df_reviews_bronze.groupBy("comentario").count().orderBy(F.desc("count")).limit(15))
print("comentário NULL:", df_reviews_bronze.where(F.col("comentario").isNull()).count(),
      "| só espaços/vazio:", df_reviews_bronze.where(F.trim("comentario") == "").count())

# Duplicatas integrais (filme, usuário, nota, comentário)
print("combinações distintas:", df_reviews_bronze.select("id", "nome", "nota", "comentario").distinct().count())

### 3.2 Transformação
**Decisões:**
- **Nota:** mesma lógica da seção 2 (só converte o que tem formato numérico; fora de 0 a 10 → `NULL`).
- **Comentário ausente:** `NULL`, vazio, só espaços **e** os textos literais `null`/`none`/`nan` viram `"Sem comentário"`, texto explícito e padronizado, como pede o enunciado.
- **Deduplicação *depois* da limpeza.** Dois registros que só diferiam na sujeira (por exemplo, `NULL` e `"null"` no comentário) passam a ser iguais depois de padronizados; deduplicar antes deixaria os dois.

In [0]:
# Nota: converte só se o valor inteiro tem cara de número; fora de 0-10 vira NULL
t = F.trim("nota")
nota_num = F.when(t.rlike(r"^-?[0-9]+([.,][0-9]+)?$"), F.regexp_replace(t, ",", ".").cast("double"))
nota_valida = F.when((nota_num >= 0) & (nota_num <= 10), nota_num)

# Comentário: NULL, vazio, só espaços ou o texto literal "null" -> "Sem comentário"
com = F.trim("comentario")
comentario = F.when(com.isNull() | (com == "") | (F.lower(com).isin("null", "none", "nan")),
                    F.lit("Sem comentário")).otherwise(F.col("comentario"))

df_reviews_silver = (df_reviews_bronze.where(F.col("id").rlike(ID_NUMERICO))
    .select(
        F.col("id").alias("id_filme"),
        F.trim("nome").alias("nome_usuario"),
        nota_valida.alias("nota_usuario"),
        comentario.alias("comentario_usuario"))
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"]))

df_reviews_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_avaliacoes_usuarios")

### 3.3 Validação

In [0]:
s = spark.table("silver.tb_avaliacoes_usuarios")

# FAIL: sem duplicatas integrais, notas na escala, comentário sempre preenchido
assert s.count() == s.dropDuplicates().count(), "Há avaliações duplicadas"
assert s.where("nota_usuario < 0 OR nota_usuario > 10").count() == 0, "Nota fora de 0-10"
assert s.where("comentario_usuario IS NULL OR trim(comentario_usuario) = ''").count() == 0, "Comentário vazio"

# QUIET: só reporta
print("Avaliações:", s.count(), "| notas nulas:", s.where("nota_usuario IS NULL").count())
display(s.groupBy("comentario_usuario").count().orderBy(F.desc("count")).limit(5))

---
# 4 · `tb_credits_and_tags`: gêneros e pessoas/empresas
Uma única tabela Bronze alimenta **duas** tabelas Silver (`tb_generos` e `tb_pessoas_empresas`). Ela é lida e filtrada **uma vez**, e as duas seções partem do mesmo DataFrame.

Todas as colunas desta origem são **multivaloradas** (`"Action, Adventure, Comedy"`), então o tratamento central é *split + explode*.

## O que a exploração da origem mostrou
- **Três separadores** convivem na coluna `genres`: vírgula, ponto e vírgula e barra vertical (`Comedy|Drama`). O enunciado cita só os dois primeiros; ignorar a barra deixaria gêneros colados.
- **Domínio de gêneros fechado:** os valores frequentes são exatamente os 19 gêneros do TMDB. O que está fora (caminhos de imagem como `/8hI7...jpg`, números como `3.339`, pedaços de sinopse, nomes de pessoas e empresas) é lixo de *column shift*.
- **Marcadores de ausência** nas colunas de pessoas: `null`, `[]`, `Nenhum` e `N/A`. Em `production_companies` aparece até um gênero (`Documentary`) deslocado.

In [0]:
df_credits_bronze = spark.table("bronze.tb_credits_and_tags")

# Chave válida: id numérico. Esta é a base das duas tabelas Silver desta seção.
df_credits = df_credits_bronze.where(F.col("id").rlike(ID_NUMERICO))

### 4.0 Exploração dos gêneros
Mede os separadores e lista os valores **mais frequentes** (o domínio real) e os **mais raros** (onde o lixo aparece). Aqui o split já usa os três separadores, como na regra final.

In [0]:
print("Linhas:", df_credits_bronze.count(), "| ids distintos:", df_credits_bronze.select("id").distinct().count())
print("ids não numéricos:", df_credits_bronze.where(~F.col("id").rlike(ID_NUMERICO) | F.col("id").isNull()).count())

# Quantas linhas usam cada separador em genres
g = F.col("genres")
display(df_credits_bronze.agg(
    F.count(F.when(g.contains(","), 1)).alias("com_virgula"),
    F.count(F.when(g.contains(";"), 1)).alias("com_ponto_virgula"),
    F.count(F.when(g.contains("|"), 1)).alias("com_barra"),
    F.count(F.when(g.isNull(), 1)).alias("nulos")))

# Valores de gênero depois do split: mais frequentes (domínio) e mais raros (lixo)
valores_genero = separar_e_explodir(df_credits_bronze, "genres", "valor")
display(valores_genero.groupBy("valor").count().orderBy(F.desc("count")).limit(40))
display(valores_genero.groupBy("valor").count().orderBy("count").limit(40))

## 4.1 `silver.tb_generos`
**Grão:** um registro por filme e por gênero.

**Decisões:**
- **Split por `,` `;` `|`** e *explode*, um gênero por linha.
- **Domínio fechado via `inner join`.** Em vez de tentar descrever o lixo (números, caminhos, frases, nomes), aceitamos **somente** os 19 gêneros conhecidos. É a regra mais simples e a mais segura, porque o lixo é imprevisível, mas o domínio válido não é.
- **Grafia oficial na saída.** A comparação é em minúsculas e o resultado usa a grafia da lista, o que também padroniza a caixa.
- **`distinct`:** elimina repetições por cópias duplicadas do mesmo filme.

In [0]:
# Tabela de referência: chave em minúsculas -> grafia oficial
ref_generos = spark.createDataFrame([(g, g.lower()) for g in GENEROS], "genero string, chave string")

generos_limpos = (separar_e_explodir(df_credits, "genres", "valor")
                  .select("id_filme", F.lower("valor").alias("chave")))

# Inner join com o domínio: descarta vazios, números, caminhos de imagem, textos descritivos e nomes deslocados (column shift)
df_generos_silver = (generos_limpos.join(F.broadcast(ref_generos), "chave", "inner")
                     .select("id_filme", "genero").distinct())

df_generos_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_generos")

### 4.1.1 Validação
**FAIL:** todo gênero pertence ao domínio e não há repetição de (filme, gênero). **QUIET:** distribuição por gênero e quantos filmes têm ao menos um gênero.

In [0]:
s = spark.table("silver.tb_generos")

# FAIL: só gêneros do domínio e sem repetição de (filme, gênero)
assert {r[0] for r in s.select("genero").distinct().collect()} <= set(GENEROS), "Gênero fora do domínio"
assert s.count() == s.dropDuplicates(["id_filme", "genero"]).count(), "Pares (filme, gênero) repetidos"

# QUIET: só reporta
display(s.groupBy("genero").count().orderBy(F.desc("count")))
print("Filmes com ao menos um gênero:", s.select("id_filme").distinct().count())

### 4.1.2 Exploração de pessoas e empresas
Para as colunas `cast`, `directors`, `writers` e `production_companies`, cada valor é classificado como `ok` ou como um tipo de lixo (vazio, marcador de ausência, numérico, caminho de imagem, texto longo, aspas ou barra). Os valores `ok` mais frequentes mostram se ainda sobrou intruso (como um gênero na coluna de produtoras). O resultado alimenta os filtros de 4.2.


In [0]:
def classificar(coluna):
    # Mesmo split da transformação, para a exploração refletir a regra final
    valores = separar_e_explodir(df_credits_bronze, coluna, "v")
    return valores.withColumn("tipo",
        F.when(F.col("v") == "", "vazio")
         .when(F.lower("v").isin(PLACEHOLDERS), "placeholder")
         .when(F.col("v").rlike(r"^[0-9.,\-\s]+$"), "numerico")
         .when(F.col("v").rlike(r"^/|\.(jpg|png)$"), "caminho_imagem")
         .when(F.length("v") > 80, "texto_longo")
         .when(F.col("v").rlike(r'[\\"]'), "aspas_ou_barra")
         .otherwise("ok"))

for coluna in ["cast", "directors", "writers", "production_companies"]:
    print("=" * 60, coluna, "| linhas com '|':", df_credits_bronze.where(F.col(coluna).contains("|")).count())
    classificados = classificar(coluna)
    display(classificados.groupBy("tipo").count().orderBy(F.desc("count")))
    # Os valores "ok" mais frequentes: se houver gênero ou lixo, ele aparece aqui
    display(classificados.where("tipo = 'ok'").groupBy("v").count().orderBy(F.desc("count")).limit(25))

## 4.2 `silver.tb_pessoas_empresas`
**Grão:** um registro por filme, entidade e tipo. Dimensão **unificada** de quatro tipos de entidade.

| Coluna de origem | `tipo_entidade` |
|---|---|
| `cast` | `Ator` |
| `directors` | `Diretor` |
| `writers` | `Roteirista` |
| `production_companies` | `Produtora` |

**Decisões:**
- **Uma função, quatro colunas, `union`.** O mesmo split + explode é aplicado às quatro colunas e o resultado é unido com o tipo correspondente, em vez de escrever quatro blocos quase iguais.
- **Filtro de nome plausível.** Diferente dos gêneros, aqui **não existe domínio fechado** (há milhões de nomes possíveis), então o lixo é descartado por *padrões*: marcador de ausência, gênero deslocado, só números, caminho de imagem, resto de aspas ou barra, tamanho fora de 2 a 80 caracteres e frases com mais de 8 palavras (sinopses).
- **Capitalização conservadora.** Aplica `initcap` somente a nomes **todo em minúsculas** ou **todo em MAIÚSCULAS com mais de uma palavra**. Nomes em caixa mista (`DC Films`, `McDonald`) e siglas de uma palavra (`BBC`) são preservados, porque `initcap` as estragaria (`Dc Films`, `Bbc`).
- **`distinct`** elimina duplicatas, inclusive as que só diferiam na caixa antes da padronização.

In [0]:
# Mapeamento exigido pelo enunciado: coluna de origem -> tipo_entidade
tipos = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista",
         "production_companies": "Produtora"}

def extrair(coluna, tipo):
    # Split + explode de uma coluna, com o tipo da entidade
    return separar_e_explodir(df_credits, coluna, "nome").withColumn("tipo_entidade", F.lit(tipo))

todos = reduce(lambda a, b: a.unionByName(b), [extrair(k, v) for k, v in tipos.items()])

# Nome plausível: descarta resíduos do column shift
nome = F.col("nome")
valido = (
    (F.length(nome) >= 2) & (F.length(nome) <= 80)
    & ~F.lower(nome).isin(PLACEHOLDERS + GENEROS_LOWER)   # ausência e gêneros deslocados
    & ~nome.rlike(r"^[0-9.,\-\s]+$")                       # valores numéricos
    & ~nome.rlike(r"^/|\.(jpg|jpeg|png)$")                 # caminhos de imagem
    & ~nome.rlike(r'[\\"]')                                # sobras de texto deslocado
    & (F.size(F.split(nome, r"\s+")) <= 8)                 # frases e sinopses
)

minusculo = nome == F.lower(nome)
maiusculo_multi = (nome == F.upper(nome)) & nome.contains(" ")
nome_padrao = F.when(minusculo | maiusculo_multi, F.initcap(nome)).otherwise(nome)

df_pessoas_silver = (todos.where(valido)
         .select("id_filme", nome_padrao.alias("nome_entidade"), "tipo_entidade")
         .distinct())   

df_pessoas_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_pessoas_empresas")

### 4.2.1 Validação
**FAIL:** nenhum gênero sobrou como nome; os quatro tipos de entidade estão presentes; nenhum nome vazio. **QUIET:** volume por tipo e os nomes mais frequentes de cada um. Se sobrou lixo, ele aparece nesses topos.

In [0]:
s = spark.table("silver.tb_pessoas_empresas")

# FAIL: nenhum gênero pode ter sobrado como nome, e os 4 tipos devem existir
assert s.where(F.lower("nome_entidade").isin(GENEROS_LOWER)).count() == 0, "Gênero sobrou como nome"
assert {r[0] for r in s.select("tipo_entidade").distinct().collect()} == set(tipos.values()), "Tipo de entidade ausente"
assert s.where("nome_entidade IS NULL OR trim(nome_entidade) = ''").count() == 0, "Nome vazio"

# QUIET: volume por tipo e topo de cada tipo
display(s.groupBy("tipo_entidade").agg(F.count("*").alias("registros"),
                                       F.countDistinct("nome_entidade").alias("nomes_distintos")))
for tipo in tipos.values():
    display(s.where(F.col("tipo_entidade") == tipo).groupBy("nome_entidade").count()
              .orderBy(F.desc("count")).limit(10))